In [1]:
# 1) Design model (input, output size, forward pass) 
# 2) Construct loss and optimizer 
# 3) Training loop 
# - forward pass: compute prediction and loss
# - backward pass: gradients 
# - update weights 

In [6]:
import torch
import torch.nn as nn 
import numpy as np 
from sklearn import datasets
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# 0. prepare data, binary classification 
bc = datasets.load_breast_cancer()

X, y = bc.data, bc.target

n_samples, n_features = X.shape
print(n_samples, n_features)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1234) 

# scale, make features 0 mean and unit variance
sc = StandardScaler()

# makes the mean 0 and the variance 1
X_train = sc.fit_transform(X_train)

# use the same transformation for the test set without fitting it
X_test = sc.transform(X_test)

X_train = torch.from_numpy(X_train.astype(np.float32))
X_test = torch.from_numpy(X_test.astype(np.float32))
y_train = torch.from_numpy(y_train.astype(np.float32))
y_test = torch.from_numpy(y_test.astype(np.float32))

# column vector 
y_train = y_train.view(y_train.shape[0], 1)
y_test = y_test.view(y_test.shape[0], 1)

# 1. model 
# f = wx + b, sigmoid at the end
class LogisticRegression(nn.Module): 
    def __init__(self, n_input_features): 
        super(LogisticRegression, self).__init__()
        self.linear = nn.Linear(n_input_features, 1) 

    def forward(self, x): 
        y_predicted = torch.sigmoid(self.linear(x)) 
        return y_predicted
    
model = LogisticRegression(n_features) 

# 2. loss and optimizer, binary cross entropy loss and SGD
criterion = nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

# 3. training loop 
num_epochs = 100

for epoch in range(num_epochs): 
    # forward pass and loss calc 
    y_predicted = model(X_train) 

    # predicted and actual 
    loss = criterion(y_predicted, y_train) 

    # backward pass
    loss.backward()

    # update weigths 
    optimizer.step()
    
    # empty the gradients
    optimizer.zero_grad()

    if(epoch % 10 == 0): 
        print(f'epoch: {epoch}, loss: {loss.item():.4f}')

# eval without tracking history 
with torch.no_grad(): 
    # predicted values from test, 0 - 1 returned by sigmoid 
    y_predicted = model(X_test)

    # convert to binary class 0 / 1
    y_predicted_cls = y_predicted.round()

    # correct / total 
    acc = y_predicted_cls.eq(y_test).sum() / float(y_test.shape[0])
    print(f'accuracy: {acc.item():.4f}')
    

569 30
epoch: 0, loss: 0.4196
epoch: 10, loss: 0.3730
epoch: 20, loss: 0.3388
epoch: 30, loss: 0.3126
epoch: 40, loss: 0.2916
epoch: 50, loss: 0.2744
epoch: 60, loss: 0.2600
epoch: 70, loss: 0.2477
epoch: 80, loss: 0.2370
epoch: 90, loss: 0.2276
accuracy: 0.8947
